# Scenario 0 - Text Field Combination Ablation

**Tujuan**: membandingkan tujuh kombinasi field teks (`title`, `abstract`, `keyword`, dan kombinasinya) untuk menentukan representasi input yang paling tepat sebelum pipeline topic modeling utama dijalankan.


## 1. Instalasi Dependency


In [ ]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn

## 2. Setup Konfigurasi dan Library


In [ ]:
import os, re, ast, json, random, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED            = 42
CPU_COUNT       = os.cpu_count() or 2
WORKERS         = max(1, CPU_COUNT - 1)
GPU_COUNT       = torch.cuda.device_count()
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_DIR       = Path("/kaggle/input")
OUTPUT_DIR      = Path("/kaggle/working/outputs_scenario0_field_combination")
EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"

random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[env] = str(CPU_COUNT)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print(f"CPU={CPU_COUNT} | WORKERS={WORKERS} | GPU={GPU_COUNT}x ({DEVICE})")

## 3. Load Dataset


In [ ]:
def find_dataset(base):
    matches = list(base.rglob("dataset_pubmed_clean.parquet"))
    if matches: return matches[0]
    raise FileNotFoundError("dataset_pubmed_clean.parquet not found in /kaggle/input")

df = pd.read_parquet(find_dataset(INPUT_DIR))
print(f"Shape: {df.shape}")
print(f"Cols : {df.columns.tolist()}")

## 4. Auto-detect Kolom


In [ ]:
def get_col(df, names, required=False):
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m: return m[n.lower()]
    if required: raise ValueError(f"Column not found: {names}")
    return None

col_pmid = get_col(df, ["pmid", "paper_id", "id"])
col_title = get_col(df, ["title", "article_title", "paper_title"])
col_abs   = get_col(df, ["abstract", "clean_abstract"], required=True)
col_kw    = get_col(df, ["keyword", "keywords", "mesh_terms", "mesh_term", "mesh", "mesh_headings", "clean_tags", "raw_tags", "tags"])
col_lbl   = get_col(df, ["final_labels"])

print(f"title   -> {col_title}")
print(f"abstract-> {col_abs}")
print(f"keyword -> {col_kw}")
print(f"labels  -> {col_lbl}")

if col_title is None or col_kw is None:
    raise ValueError(
        "Kolom title atau keyword tidak ditemukan. "
        f"Kolom tersedia: {df.columns.tolist()}"
    )

## 5. Prepare Base Data


In [ ]:
def parse_labels(v):
    if isinstance(v, (list, np.ndarray)): return list(v)
    try:
        if pd.isna(v): return []
    except (TypeError, ValueError): pass
    s = str(v).strip()
    try:
        p = ast.literal_eval(s)
        if isinstance(p, list): return [str(x) for x in p]
    except: pass
    return [x.strip() for x in re.split(r"[;,|]", s) if x.strip()]

def field_to_text(v):
    """Normalisasi field (termasuk list/array keyword) jadi string."""
    if isinstance(v, (list, np.ndarray)):
        return " ".join(str(x) for x in v)
    try:
        if pd.isna(v): return ""
    except (TypeError, ValueError): pass
    return str(v)

def light_clean(text):
    t = str(text).lower()
    t = re.sub(r"<.*?>", " ", t)
    return re.sub(r"\s+", " ", t).strip()

base = pd.DataFrame({
    "paper_id"   : df[col_pmid].astype(str) if col_pmid else np.arange(len(df)).astype(str),
    "title_raw"  : df[col_title].apply(field_to_text),
    "abstract_raw": df[col_abs].apply(field_to_text),
    "keyword_raw": df[col_kw].apply(field_to_text),
    "label_list" : df[col_lbl].apply(parse_labels) if col_lbl else [[] for _ in range(len(df))],
})
base["primary_label"] = base["label_list"].apply(lambda x: x[0] if x else "Unknown")
base = base[base["primary_label"] != "Unknown"].reset_index(drop=True)

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    base["title_clean"]    = list(ex.map(light_clean, base["title_raw"], chunksize=500))
    base["abstract_clean"] = list(ex.map(light_clean, base["abstract_raw"], chunksize=500))
    base["keyword_clean"]  = list(ex.map(light_clean, base["keyword_raw"], chunksize=500))

# Filter: minimal abstract harus ada isi
base = (base[base["abstract_clean"].str.split().str.len() >= 5]
            .drop_duplicates("abstract_clean")
            .reset_index(drop=True))

y_true = base["primary_label"].tolist()
print(f"Rows={len(base)} | Labels={base['primary_label'].nunique()}")
print(f"\nSample title   : {base['title_clean'].iloc[0][:100]}")
print(f"Sample keyword : {base['keyword_clean'].iloc[0][:100]}")

## 6. Bangun 7 Kombinasi Teks


In [ ]:
def combine(row, fields):
    parts = [row[f"{f}_clean"] for f in fields if row[f"{f}_clean"]]
    return " ".join(parts).strip()

COMBINATIONS = {
    "title_only"             : ["title"],
    "abstract_only"          : ["abstract"],
    "keyword_only"           : ["keyword"],
    "title_abstract"         : ["title", "abstract"],
    "title_keyword"          : ["title", "keyword"],
    "abstract_keyword"       : ["abstract", "keyword"],
    "title_abstract_keyword" : ["title", "abstract", "keyword"],
}

combo_docs = {}
for name, fields in COMBINATIONS.items():
    texts = base.apply(lambda r: combine(r, fields), axis=1)
    valid_mask = texts.str.split().str.len() >= 3  # minimal 3 kata
    combo_docs[name] = {
        "docs"  : texts[valid_mask].tolist(),
        "y_true": base.loc[valid_mask, "primary_label"].tolist(),
        "n"     : int(valid_mask.sum()),
    }
    print(f"{name:24s} -> {combo_docs[name]['n']} docs | sample: {combo_docs[name]['docs'][0][:80]}")

## 7. Evaluation Setup


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score
from scipy.optimize import linear_sum_assignment
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

BIOMED_STOP = {
    "patient","patients","study","studies","result","results",
    "method","methods","conclusion","conclusions","background",
    "objective","objectives","data","analysis","significant",
    "significantly","associated","compared","comparison",
    "included","including","use","used","using","group","groups",
    "showed","shown"
}
STOP_ALL = set(STOPWORDS) | BIOMED_STOP

def tokenize(text):
    return [t for t in simple_preprocess(text, deacc=True, min_len=3) if t not in STOP_ALL]

def topic_uniqueness(tw):
    all_w = [w for t in tw for w in t]
    return round(len(set(all_w)) / len(all_w), 4) if all_w else 0.0

def hungarian_f1(y_true, y_topic):
    labels = sorted(set(y_true))
    topics = sorted(set(y_topic))
    l2i = {l: i for i, l in enumerate(labels)}
    t2i = {t: i for i, t in enumerate(topics)}
    yt  = np.array([l2i[y] for y in y_true])
    yp  = np.array([t2i[y] for y in y_topic])
    M   = np.zeros((len(topics), len(labels)))
    for t in topics:
        for l in labels:
            M[t2i[t], l2i[l]] = f1_score((yt == l2i[l]).astype(int),
                                          (yp == t2i[t]).astype(int), zero_division=0)
    ri, ci = linear_sum_assignment(-M)
    t2l = {topics[r]: labels[c] for r, c in zip(ri, ci)}
    fb  = pd.Series(y_true).mode()[0]
    return round(f1_score(y_true, [t2l.get(t, fb) for t in y_topic], average="macro", zero_division=0), 4)

def extract_topic_words(tm, topn=10):
    return [
        [w for w, _ in tm.get_topic(t)[:topn] if isinstance(w, str) and len(w) > 1]
        for t in sorted(x for x in tm.get_topics() if x != -1)
    ]

vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | BIOMED_STOP), ngram_range=(1, 2), min_df=1
)

print("Evaluation setup ready.")

## 8. Run Pipeline per Kombinasi


In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

model_emb = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
model_emb.max_seq_length = 512

results = []

for name, payload in combo_docs.items():
    print(f"\n{'='*50}")
    print(f"Kombinasi: {name} ({payload['n']} docs)")
    print(f"{'='*50}")

    docs   = payload["docs"]
    y_true = payload["y_true"]

    # Embed
    if GPU_COUNT > 1:
        pool = model_emb.start_multi_process_pool(target_devices=[f"cuda:{i}" for i in range(GPU_COUNT)])
        emb  = model_emb.encode_multi_process(docs, pool, batch_size=128, show_progress_bar=False)
        model_emb.stop_multi_process_pool(pool)
    else:
        emb = model_emb.encode(docs, batch_size=128, show_progress_bar=False,
                               convert_to_numpy=True, normalize_embeddings=True)
    emb = np.asarray(emb, dtype=np.float32)

    # UMAP
    reduced = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                   metric="cosine", random_state=SEED, n_jobs=1).fit_transform(emb)

    # HDBSCAN default (min_cluster_size=5)
    hdb = HDBSCAN(min_cluster_size=5, metric="euclidean",
                  cluster_selection_method="eom", prediction_data=True,
                  core_dist_n_jobs=WORKERS)
    tm = BERTopic(
        umap_model=__import__("bertopic.dimensionality", fromlist=["BaseDimensionalityReduction"]).BaseDimensionalityReduction(),
        hdbscan_model=hdb, vectorizer_model=vectorizer_model,
        top_n_words=15, calculate_probabilities=False, verbose=False
    )
    topics, _ = tm.fit_transform(docs, embeddings=reduced)
    topics = list(topics)

    # Tokenize untuk coherence
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        tok_docs = list(ex.map(tokenize, docs, chunksize=500))
    g_dict = Dictionary(tok_docs)
    g_dict.filter_extremes(no_below=5, no_above=0.5, keep_n=50_000)

    k    = len(set(topics)) - (1 if -1 in topics else 0)
    out  = round(float(np.mean(np.array(topics) == -1)), 4)
    tw   = extract_topic_words(tm, topn=10)
    cv   = round(CoherenceModel(topics=tw, texts=tok_docs, dictionary=g_dict,
                                coherence="c_v", processes=WORKERS).get_coherence(), 4) if tw else 0.0
    f1   = hungarian_f1(y_true, topics)
    uniq = topic_uniqueness(tw)
    hm   = round(2*cv*f1/(cv+f1), 4) if (cv+f1) > 0 else 0.0

    results.append({
        "combination": name, "n_docs": payload["n"], "n_topics": k,
        "outlier_rate": out, "cv": cv, "topic_uniqueness": uniq,
        "hungarian_f1": f1, "harmonic_mean_cv_f1": hm
    })
    print(f"K={k} | Outlier={out} | Cv={cv} | F1={f1} | HM={hm}")

## 9. Ranking & Save


In [ ]:
results_df = pd.DataFrame(results)

print("=== Ranking by Harmonic Mean (Cv, F1) ===")
display(results_df.sort_values("harmonic_mean_cv_f1", ascending=False))

print("\n=== Ranking by Cv ===")
display(results_df.sort_values("cv", ascending=False))

print("\n=== Ranking by Hungarian F1 ===")
display(results_df.sort_values("hungarian_f1", ascending=False))

best = results_df.sort_values("harmonic_mean_cv_f1", ascending=False).iloc[0]
print("\n" + "="*50)
print(f"BEST COMBINATION: {best['combination']}")
print(f"  K={int(best['n_topics'])} | Outlier={best['outlier_rate']} | Cv={best['cv']} | F1={best['hungarian_f1']} | HM={best['harmonic_mean_cv_f1']}")
print("="*50)

results_df.to_csv(OUTPUT_DIR / "scenario0_field_combination_results.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}")